<a href="https://colab.research.google.com/github/cgm2179/indoor-walk-test/blob/main/Physics%20Engine/3D%20Map%20Physics/SIM%20V2/Indoor/Multiband_resumable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIM V2 3D — Resumable high-detail 7-band training (Phase B + C · A100)

Generate conformal **18-channel** training data for **all 7 bands** at **λ/10** full-detail resolution
and train ONE pure-JAX 3-D U-Net (**base=48, ~13 M params**) on the lot, with a **Helmholtz-residual
physics loss**. Built to **survive Colab timeouts**: everything (shards, checkpoints, weights) lives on
**Google Drive**, every band is **marked done**, training **checkpoints every 5 epochs**. On a restart
just **Runtime → Run all** — Phase B skips finished bands, Phase C resumes from the last checkpoint.

- **Bands**: `LTE_B71_617 · LTE_B13_751 · LTE_B2_1960 · NR_n41_2506 · NR_n77_3700 · WiFi_2G4 · WiFi_5G`
- **Detail**: per-Tx **mesh bake @ λ/10** with adaptive super (~5 mm effective-medium floor) — full CAD
  detail at the solve resolution. The 329 MB OBJ **ships in the repo gzipped (76 MB)** and is auto-
  decompressed in §0b — no upload needed. (Or set `MESH_BAKE=False` for the coarser frozen-geometry path.)
- **GPU**: pick an **A100** (Runtime → Change runtime type). **CuPy** runs the voxelizer's closing +
  distance-transform on-device and **JAX/XLA** runs the FDTD (`simulate_jax`) — the whole pipeline is GPU.
- **Input channels (18)**: 6 material one-hot + Tx + freq + log-dist + εr + σ + SDF + 3 normals + the
  **antenna current map |Jx|,|Jy|,|Jz|**. **Output**: complex scalar field (Re, Im).

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # let CuPy (voxelizer) share the GPU with JAX
# === Environment bootstrap: Colab or local ===
import sys
from pathlib import Path
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BRANCH = "sim-v2-multiband-resumable"   # branch carrying the high-detail pipeline + frac.npy
    REPO = Path("/content/indoor-walk-test")
    if not REPO.exists():
        !git clone --depth 1 -b {BRANCH} https://github.com/cgm2179/indoor-walk-test.git "{REPO}"
else:
    here = Path.cwd()
    REPO = next((p for p in [here, *here.parents] if (p / "Physics Engine").is_dir()),
                Path("/Users/cameronmickle/Documents/Indoor_Walk_Test_7-7"))
SIMV2 = REPO / "Physics Engine" / "3D Map Physics" / "SIM V2"
sys.path.insert(0, str(SIMV2))
print("IN_COLAB:", IN_COLAB, "| REPO:", REPO)

## 0a · Persistence — mount Google Drive
**This is what makes it resumable.** All shards + checkpoints + weights live in one Drive folder that
outlives the runtime; a fresh session re-mounts it and the pipeline picks up where it stopped.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = Path("/content/drive/MyDrive/simv2_multiband")   # <- your Drive; survives timeouts
else:
    PERSIST = SIMV2 / "multiband_run"
PERSIST.mkdir(parents=True, exist_ok=True)
SHARDS  = PERSIST / "shards"        # per-band 18-ch shards + done-markers
CONF    = PERSIST / "conformal"     # per-band composed grids (frozen-geometry fallback only)
CKPT    = PERSIST / "unet3d_ckpt.pkl"   # (params + optax state + epoch) — the resume file
WEIGHTS = PERSIST / "unet3d_jax.npz"    # final exported weights
print("persist ->", PERSIST)
for p in sorted(PERSIST.rglob("*"))[:20]:
    print("  ", p.relative_to(PERSIST))

## 0b · Full-detail geometry — decompress the committed OBJ + enable CuPy
`MESH_BAKE=True` voxelizes the real CAD per transmitter at λ/10, so the OBJ must be on the node. It
**ships in the repo gzipped** (`SIM V2/data/floor7.obj.gz`, 76 MB) and this cell **auto-decompresses**
it once — no upload. (A Drive copy at `MyDrive/simv2_multiband/…obj` is used as a fallback if present.)
**CuPy** (pre-installed on Colab CUDA) runs the voxelizer's closing + distance-transform on the GPU;
the JAX/XLA FDTD (`simulate_jax`) is already on the GPU.

In [ ]:
import voxelize_conformal as VC, gzip, shutil
OBJ_GZ    = SIMV2 / "data" / "floor7.obj.gz"                            # committed (gzipped, 76 MB)
OBJ_LOCAL = Path("/content/floor7.obj") if IN_COLAB else (SIMV2 / "data" / "floor7.obj")
OBJ_DRIVE = PERSIST / "3ff8432c-980b-41b6-9045-c09dbe1d74cc copy.obj"   # optional Drive fallback
if OBJ_GZ.exists() and not OBJ_LOCAL.exists():
    print("decompressing committed OBJ (once) …")
    with gzip.open(OBJ_GZ, "rb") as f, open(OBJ_LOCAL, "wb") as g:
        shutil.copyfileobj(f, g)
if OBJ_LOCAL.exists():
    VC.OBJ = str(OBJ_LOCAL)
elif OBJ_DRIVE.exists():
    VC.OBJ = str(OBJ_DRIVE)
else:
    print("⚠ no OBJ found — set MESH_BAKE=False for the frozen-geometry path")
print("OBJ ->", VC.OBJ)
# CuPy backend for the voxelizer (closing + distance transform on-device); JAX already runs the FDTD on GPU
try:
    import cupy; VC.USE_GPU = VC._HAS_CUPY
except Exception:
    VC.USE_GPU = False
print("voxelizer backend:", "CuPy/GPU" if VC.USE_GPU else "NumPy/CPU")

In [ ]:
import numpy as np, glob, time, json
import jax
import bootstrap  # noqa: F401  (SIM V2 path shim)
import fw_dataset3d as D
import unet3d_train_jax as T
print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

## 1 · Configuration

In [ ]:
BANDS = ["LTE_B71_617", "LTE_B13_751", "LTE_B2_1960", "NR_n41_2506",
         "NR_n77_3700", "WiFi_2G4", "WiFi_5G"]
N_TX       = 20      # transmitters per band
BOXES_PER  = 150     # 24³ boxes cropped per Tx (stratified over x/y/z) -> 7*20*150 = 21,000 boxes
EPOCHS     = 120
BASE       = 48      # U-Net base width (~13 M params)
BS         = 32      # batch size (A100)
SUPER      = "auto"  # adaptive geometry supersampling (~5 mm floor): 8 @ low bands, 2-3 @ high
MESH_BAKE  = True    # per-Tx mesh bake @ λ/10 from the OBJ (full detail). False = frozen-geometry (coarser)
ANTENNA    = True    # 18-ch input with |Jx|,|Jy|,|Jz|; ~50% of Tx are directional (panel patterns)
PHYS_WT    = 0.05    # Helmholtz-residual loss weight (0 = pure MSE)
CKPT_EVERY = 2       # checkpoint to Drive every N epochs (a kill loses ≤N epochs; set 1 for max safety)

GEOM = REPO / "Physics Engine" / "3D Map Physics" / "SIM V1 3D" / "conformal" / "LTE_B71_617"  # frac.npy (fallback)
for b in BANDS:
    print(f"  {b:14s} region={D.BAND_PLAN[b][0]}m  npw=λ/{D.BAND_PLAN[b][1]:.0f}")

## 2 · Corpus + rough time (measure epoch 1 for the real number)

In [ ]:
dev = jax.default_backend()
boxes = len(BANDS) * N_TX * BOXES_PER
steps = (boxes + BS - 1) // BS * EPOCHS
print(f"runtime={dev}   corpus={boxes:,} boxes   {steps:,} train-steps (base={BASE}, bs={BS})")
print(f"  Phase B (mesh bake @ λ/10, 7 bands × {N_TX} Tx): ~1–3 h on A100 (CuPy voxelize + JAX FDTD)")
if dev == "cpu":
    print("  Phase C: base-48 on 63k boxes is INFEASIBLE on CPU — switch to an A100 GPU runtime.")
else:
    print(f"  Phase C ({EPOCHS} ep): rough ~10–40 h on A100 — the first epoch prints its wall-time,")
    print( "           multiply by EPOCHS for the real estimate. Checkpointed → spread across sessions.")
    print(f"  Shrink knobs if needed: EPOCHS↓, BOXES_PER↓, or N_TX↓.")

## 3 · Phase B — generate all 7 bands (resumable, marked)

Per band: solve `N_TX` transmitters (mesh bake @ λ/10 + adaptive super; ~50% directional), crop 300
stratified 18-channel boxes, write shards to Drive, drop a `dataset_meta.json` **done-marker**.
Re-running **skips** finished bands — a timeout only ever costs the band in flight.

In [9]:
t0 = time.time()
D.generate_bands(BANDS, out_root=SHARDS, conformal_root=CONF, geom_dir=str(GEOM),
                 n_tx=N_TX, boxes_per=BOXES_PER, seed=1, mesh_bake=MESH_BAKE, super=SUPER,
                 antenna=ANTENNA, directional_frac=0.5)
done = [b for b in BANDS if (SHARDS / b / "dataset_meta.json").exists()]
print(f"\nPhase B: {len(done)}/{len(BANDS)} bands done in {time.time()-t0:.0f}s  -> {done}")

[skip] LTE_B71_617: shards already generated
[skip] LTE_B13_751: shards already generated
[skip] LTE_B2_1960: shards already generated
[skip] NR_n41_2506: shards already generated
[mesh] NR_n77_3700: per-Tx mesh bake @ λ/10 + adaptive super (full detail)
[gen ] NR_n77_3700: n_tx=20 boxes=150 region=2.5m npw=λ/10
  [skip] shard 000 already done
  [skip] shard 001 already done
  [skip] shard 002 already done
  [skip] shard 003 already done
  [skip] shard 004 already done
  [skip] shard 005 already done
  [skip] shard 006 already done
  [skip] shard 007 already done
  [skip] shard 008 already done
  [skip] shard 009 already done
  [skip] shard 010 already done
  [skip] shard 011 already done
  two-tier: 5% of faces = sub-wavelength clutter (<20 verts) → homogenized
  shard 012 boxes=150 x(18, 24, 24, 24) [iso 18-ch]
  two-tier: 5% of faces = sub-wavelength clutter (<20 verts) → homogenized
  shard 013 boxes=150 x(18, 24, 24, 24) [iso 18-ch]
  two-tier: 5% of faces = sub-wavelength clutter

## 4 · Phase C — train the U-Net on all bands (resumable)

`train_resumable` loads every band's shards and every 5 epochs saves `(params + optax state + epoch)`
to `CKPT` on Drive; a fresh session reloads it and continues from the saved epoch. Loss = MSE +
`PHYS_WT`·Helmholtz-residual (∇²E+k²εr·E, isotropic samples only). Raise `EPOCHS` and re-run to train
further — it resumes, never restarts.

In [10]:
T.train_resumable(str(SHARDS), total_epochs=EPOCHS, base=BASE, bs=BS,
                  ckpt=str(CKPT), ckpt_every=CKPT_EVERY, out=str(WEIGHTS), phys_weight=PHYS_WT)
print("weights ->", WEIGHTS)

data x(21000, 18, 24, 24, 24) y(21000, 2, 24, 24, 24)  cin=18 cout=2  base=48  device=gpu  boxes=21000  phys_weight=0.05  iso_frac=0.45
[fresh ] training 0/120
  ep  0/120  mse=0.05209  phys=0.01161  78.5s/ep  ·  119 epochs left  ·  ETA 2.60 h (≈Sat 09:04)
  ep  1/120  mse=0.04597  phys=0.01974  45.1s/ep  ·  118 epochs left  ·  ETA 88.6 min (≈Sat 07:58)
    [ckpt] epoch 2 -> unet3d_ckpt.pkl
  ep  2/120  mse=0.04399  phys=0.02341  45.0s/ep  ·  117 epochs left  ·  ETA 87.8 min (≈Sat 07:58)
  ep  3/120  mse=0.04280  phys=0.02606  45.2s/ep  ·  116 epochs left  ·  ETA 87.2 min (≈Sat 07:58)
    [ckpt] epoch 4 -> unet3d_ckpt.pkl
  ep  4/120  mse=0.04267  phys=0.02684  45.2s/ep  ·  115 epochs left  ·  ETA 86.5 min (≈Sat 07:58)
  ep  5/120  mse=0.04195  phys=0.02939  45.1s/ep  ·  114 epochs left  ·  ETA 85.7 min (≈Sat 07:58)
    [ckpt] epoch 6 -> unet3d_ckpt.pkl
  ep  6/120  mse=0.04174  phys=0.02942  45.1s/ep  ·  113 epochs left  ·  ETA 85.0 min (≈Sat 07:58)
  ep  7/120  mse=0.04099  phys=0.03

## 5 · Phase D — validate vs a fresh FDTD (optional)

In [1]:
T.validate(str(WEIGHTS), "LTE_B71_617", npw=10.0, region_m=6.0)   # envelope dB RMSE / Spearman / coherence

NameError: name 'T' is not defined

## Resuming after a timeout / out of credits
**Progress lives on Google Drive, not GitHub** — you never push anything to GitHub between sessions.
GitHub only holds the code (cloned fresh each session); shards + the checkpoint sit in your Drive folder
and persist no matter how the runtime ends (timeout, out of credits, or you interrupt).
1. New session → re-select the **A100** runtime.
2. **Runtime → Run all.** Cells 0–2 re-mount Drive + re-clone the code + decompress the OBJ (seconds).
3. Phase B prints `[skip]` for every shard already on Drive — it only generates the missing transmitters
   (per-Tx resume; a kill costs at most the one Tx in flight).
4. Phase C prints `[resume] @ epoch N` and continues from the last checkpoint (loses ≤ `CKPT_EVERY` epochs).

## Knobs
- **Faster / cheaper**: `EPOCHS` 60–80, `BOXES_PER` 150, or `N_TX` 15. Still resumable.
- **No OBJ on Drive**: `MESH_BAKE=False` → frozen-geometry path (εr/σ upsampled from 0.3 m, coarser but
  needs no OBJ and no CuPy).
- **High-band memory** (3700/5500 MHz) is the tightest spot; `D.BAND_PLAN` already shrinks their region.
- **Export to ONNX**: copy the trained JAX weights into the PyTorch `UNet3DField(cin=18, base=48)` →
  `torch.onnx.export` (parity proven in `unet3d_jax.py`).